# Evidence records for auditors

A GroundLens verification does not end with `PASS` or `FAIL`. It ends with a **record**: a signed, hash-chained document that says which verifiers ran, what they measured, which policy interpreted the evidence, what was decided, and which regulatory controls that decision concerns.

This notebook shows the part of GroundLens an auditor, a compliance officer or a customer's risk team touches:

1. a log of verifications produced by an application;
2. verifying the whole chain offline: every hash, every link, every signature;
3. what happens when someone edits a record;
4. the mapping to the EU AI Act;
5. the report and the auditor's one-page guide that `groundlens report` produces.

Runs in Google Colab. The engine never opens a network connection.

In [ ]:
%pip install -q groundlens
!groundlens --version

## 1. An application writes records

Imagine a document assistant answering questions about a loan agreement. Each answer is verified under the `eu_ai_act_high_risk_v1` policy and appended to a log. The application signs with its own key, so records can be attributed to it; here we generate one and keep it in an environment variable, which is how the command line picks it up too.

In [ ]:
import os, subprocess
from pathlib import Path
from groundlens import verify, Record

os.environ["GROUNDLENS_SIGNING_KEY"] = subprocess.run(["groundlens", "keygen"], capture_output=True, text=True).stdout.strip()
SIGNING_KEY = os.environ["GROUNDLENS_SIGNING_KEY"]

CONTRACT = [
    ("loan.pdf#p1", "Loan agreement. Principal: 250,000 euros. Annual percentage rate: 4.75%. Term: 240 months."),
    ("loan.pdf#p2", "Early repayment fee: 1.00% of the outstanding balance during the first 5 years, 0.50% thereafter."),
    ("loan.pdf#p3", "The borrower must hold a home insurance policy for the whole term. Payments are due on the 25th of each month."),
]

EXCHANGES = [
    ("What is the principal?", "The principal is 250,000 euros."),
    ("What is the APR?", "The annual percentage rate is 4.75%."),
    ("How long is the term?", "The term of the loan is 240 months."),
    ("What is the early repayment fee?", "Early repayment costs 1.00% of the outstanding balance in the first 5 years and 0.50% after that."),
    ("Is insurance required?", "Yes, a home insurance policy is required for the whole term."),
    ("What is the early repayment fee after year five?", "After the fifth year the fee is 0.75% of the outstanding balance."),
    ("When are payments due?", "Payments are due on the 15th of each month."),
]

log = Path("records.jsonl")
log.unlink(missing_ok=True)
for question, answer in EXCHANGES:
    r = verify(answer, CONTRACT, question=question, locale="en", policy="eu_ai_act_high_risk_v1", signing_key=SIGNING_KEY, log=log)
    print(f"{r.decision:<7} {question}")

Two answers are wrong: the fee after year five (0.75 % instead of 0.50 %) and the payment day (15th instead of 25th). Both were caught by the exact numeric verifier, with the source number they lost to. The answer about insurance contains no numbers, so the numeric verifier has nothing to say about it; with the base bundle installed, the lexical verifier anchors its words in the sources and the policy's threshold on that score applies. Which claims may go unchecked, and what happens then, is written in the policy, not in the engine.

## 2. Verify the chain, offline

An auditor receives `records.jsonl` and nothing else. Verification recomputes every content hash, every record hash, every link to the previous record and every signature. It needs no key material from the application: the public key travels inside each record.

In [ ]:
records = Record.read_log(log)
print(Record.verify_chain(records), "records verified")
print()
print(f"{'record_id':<32} {'decision':<8} {'previous_record_hash':<26} signer")
for r in records:
    prev = (r.previous_record_hash or "—")[:23]
    print(f"{r.record_id:<32} {r.decision:<8} {prev:<26} {r.signer_public_key[:16]}…")

In [ ]:
# The same check from the command line, which is what an auditor without Python would run
# (the glv binary exposes the same command).
!groundlens record verify records.jsonl

## 3. What happens when a record is edited

Turn one `FAIL` into a `PASS` by hand and verify again. The content hash no longer matches the content, so the record is rejected; and because every record carries the hash of the previous one, removing a record from the middle of the log breaks the chain too.

In [ ]:
import json
from groundlens.record import IntegrityError

lines = log.read_text(encoding="utf-8").splitlines()
tampered = json.loads(lines[-1])
tampered["content"]["outcome"]["decision"] = "PASS"
try:
    Record.from_json(json.dumps(tampered)).verify()
except IntegrityError as e:
    print("edited record rejected:", str(e)[:120])

try:
    Record.verify_chain([Record.from_json(l) for l in lines[:3] + lines[4:]])
except IntegrityError as e:
    print("record removed from the log, chain rejected:", str(e)[:120])

## 4. The regulatory mapping

The policy, not the engine, says what a decision means for compliance. `eu_ai_act_high_risk_v1` maps `FAIL` and `REVIEW` outcomes to Art. 15(1) (accuracy, robustness) and Art. 12(1) (record keeping) of Regulation (EU) 2024/1689. The mapping is inside each record, next to the decision it applies to, and the policy hash guarantees it is the mapping that was in force at the time.

In [ ]:
for r in records:
    if r.regulatory_mapping:
        print(r.decision, r.record_id)
        for m in r.regulatory_mapping:
            print(f"   {m['framework']}  {m['article']}  {m['control']}")
        for reason in r.reasons:
            print("   →", reason)
print()
print("policy:", records[0].policy_id, records[0].policy_hash)

## 5. The report

`groundlens report` turns a log into the files an auditor actually reads: `report.md` (decisions, reasons, verifiers, policy and bundle hashes), `report.json` (the same, for tooling), the `records.jsonl` copy it was built from, and `README-auditor.md`, a one-page explanation of how to verify the package independently.

In [ ]:
!groundlens report records.jsonl --out evidence-package
print(open("evidence-package/report.md", encoding="utf-8").read())

In [ ]:
print(open("evidence-package/README-auditor.md", encoding="utf-8").read())

## What an auditor can rely on

* **Integrity**: any change to a record, or to the order of the log, is detected offline by anyone with the file.
* **Attribution**: each record is signed; the public key is in the record.
* **Reproducibility**: the same input, policy and bundle give the same content hash on any machine. Exact verifiers are bit-identical; statistical verifiers run pinned models with declared tolerances and guard bands, so a platform difference can never flip a decision.
* **Separation of duties**: the engine measured, the policy decided, and the policy is named and hashed in every record. Whoever wrote the policy owns the decision rule.

Source and documentation: [github.com/groundlens-dev/groundlens](https://github.com/groundlens-dev/groundlens).